## 1.导入相关的包

In [2]:
import torch
# nn.Module：所有网络层的基类，你自己写的模型也要继承它
# nn.Linear：全连接层（线性变换 y = xW + b）
# nn.Embedding：词嵌入层（把 token id 变成向量）
# nn.LayerNorm：层归一化
# nn.Dropout：dropout 层
# 这个两个的区别是nn是类，适合有学习参数的层，F is function 适合无参数的操作

import torch.nn as nn
import torch.nn.functional as F
# Two methods must be implemented __len__(self), __getitem(self, idx)
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass 
import math

## 不用 @dataclass 的写法：
完全等价。Python 看到 @dataclass 就自动帮你生成了：

- __init__：构造函数
- __repr__：打印时好看（print(config) 会显示所有字段）
- __eq__：两个实例能直接 == 比较



### class GPTConfig:

    def __init__(self, n_layer=12, n_head=12, n_embd=768, dropout=0.1):
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

    def __repr__(self):
        return f"GPTConfig(n_layer={self.n_layer}, n_head={self.n_head}, ...)"

    def __eq__(self, other):
        return (self.n_layer == other.n_layer and ...)

# 定义GPT的一些参数

In [3]:
@dataclass
class GPTConfig:
    block_size: int = 64        # 上下文窗口（玩具版 64；正式跑可调到 512+）
    batch_size: int = 16        # 每步多少个样本
    n_layer: int = 2            # Transformer Block 层数（玩具版 2；正式 6~12）
    n_head: int = 4             # 注意力头数（必须整除 n_embd）
    n_embd: int = 128           # 每个 token 的向量维度（玩具版 128；正式 384~768）
    hidden_dim: int = n_embd
    dropout: float = 0.1
    head_size: int = n_embd // n_head    # 128 // 4 = 32
    # gpt2 官方的 tokenizer
    vocab_size: int = 50257


先看 GPT 的两个"嵌入层"
GPT 模型的头和尾，各有一个矩阵：

1. 输入端：Token Embedding（词嵌入）

self.wte = nn.Embedding(vocab_size, n_embd)   # shape: (vocab_size, n_embd)
作用：把输入的 token id（一个整数）映射成一个 n_embd 维的向量。

输入：token_id = 42（"cat" 这个词）
输出：一个 768 维向量
2. 输出端：LM Head（语言模型头）

self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)   # weight shape: (vocab_size, n_embd)
作用：模型最后一层的输出（768 维向量）映射回 vocab_size 维，预测下一个 token 是词表中哪个词。

输入：一个 768 维向量
输出：vocab_size 维的 logits（每个词的得分）
关键观察：这两个矩阵形状一样！
两个权重矩阵的 shape 都是 (vocab_size, n_embd)。

一个是"id → 向量"
一个是"向量 → id 的得分"
本质上做的是逆操作！

Weight Tying 是什么？
让这两个矩阵共用同一份参数：


self.lm_head.weight = self.wte.weight
写完这一行，两个层指向内存中同一块权重。

为什么要这么做？
① 省参数
假设 vocab_size = 50000, n_embd = 768：

一份嵌入矩阵 = 50000 × 768 ≈ 3840 万参数
两份独立 = 7680 万参数
GPT-2 small 总共才 124M 参数，绑定后能省下接近 30% 的参数！

② 效果更好
直觉上：一个词的"输入表示"和"输出表示"应该是同一个东西。把它们绑在一起，让模型用同一份语义空间编码和解码，训练更高效，泛化更好。这个 trick 在 2016 年的论文 "Using the Output Embedding to Improve Language Models" 里被验证过。

③ 训练更稳
两份独立的话，可能输入端的 "cat" 向量和输出端的 "cat" 向量学得不一致；绑定后强制一致。

## 3. 定义GPT的结构

Attention(Q, K, V) = softmax( Q · Kᵀ / √d_k ) · V
        Q · Kᵀ
分数 = ────────
         √d_k

权重 = softmax(分数)         ← 沿最后一维做,每行和=1

输出 = 权重 · V


In [4]:
# 1. sigle head attention
class SingleHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.hidden_dim, config.head_size)
        self.value = nn.Linear(config.hidden_dim, config.head_size)
        self.query = nn.Linear(config.hidden_dim, config.head_size)
        self.head_size = config.head_size
        # attention_mask 通过 register_buffer 注册
        # 不用计算梯度，所以节约显存和内存，速度也更快
        self.register_buffer(
            "attention_mask",
            # tril (下三角)
            # block_size 是文本最大长度512
            torch.tril(
                torch.ones(config.block_size, config.block_size)                
            )
        )
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        weight = q @ k.transpose(-2, -1)
        weight = weight.masked_fill(
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        ) / math.sqrt(self.head_size)
        # 注意计算Weight的时候，除以根号下d_k
        weight = F.softmax(weight, dim=-1) 
        # dropout 要放到weight后面
        weight = self.dropout(weight)

        output = weight @ v
        return output

# 2. multi head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )    
        output = self.proj(output)
        output = self.dropout(output)
        return output
    
# 3. feed forward(MLP)
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.hidden_dim, 4 * config.hidden_dim),
            nn.GELU(),
            nn.Linear(4 * config.hidden_dim, config.hidden_dim),
            nn.Dropout(config.dropout),
        )
    def forward(self, x):
        return self.net(x)

# 4. block
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.hidden_dim)
        self.ln2 = nn.LayerNorm(config.hidden_dim)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# 5. GPT
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        # generate 时要用，存一下
        self.block_size = config.block_size
        # (embedding, position, norm, mlp, block)
        # position embedding 从0, 1, xxx, embedding 升级到rope
        # norm layer norm -> rms norm
        # mlp -> swiglu
        # mha -> gqa
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # 现在的SLM模型 会用tie weight 减少参数
        
        # 非常重要
        # linear (4->8), weight实际上的shape 是8*4
        self.token_embedding_table.weight = self.lm_head.weight

        # 把权重初始化函数应用到所有子模块上
        self.apply(self._init_weights)
               
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # 正态分布初始化
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 是输入的token ids
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)
        
        #seq 长度是这次输入的最大长度
        pos_emb = self.position_embedding_table(
            # 要确保位置编码和输入的idx在同一个设备上
            torch.arange(seq_len, device=idx.device)
        )

        # 为什么embedding 和 position可以相加
        x = token_emb + pos_emb # shape is (batch, seq_len, n_embd)
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_token):
        for _ in range(max_new_token):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # becomes (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
        

In [5]:
class MyDataset(Dataset):
    def __init__(self, path, block_size=512):
        import tiktoken
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size

        # 特殊符号分割不同的训练文本
        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"}
        )[0]

        self.encoded_data = []

        # 直接读取纯文本文件（input.txt 是莎士比亚全文，没有 JSON 结构）
        with open(path, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        # 把整段文本一次性编码成 token 序列
        full_encoded = self.enc.encode(raw_text)

        # 按 block_size 切块（每块 block_size+1，因为 x 和 y 错位一位）
        for i in range(0, len(full_encoded), self.block_size):
            chunk = full_encoded[i: i + self.block_size + 1]
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [self.eos_token] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        return self.enc.encode(text)

    def decode(self, ids):
        return self.enc.decode(ids)

In [6]:
# ============== 加载 input.txt（莎士比亚） ==============
cfg = GPTConfig()
train_dataset = MyDataset('input.txt', block_size=cfg.block_size)
print('总样本块数:', len(train_dataset))

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
print(f'train batches: {len(train_loader)}, val batches: {len(val_loader)}')

总样本块数: 5282
train batches: 298, val batches: 33


In [7]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)
model = model.to(device)

# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.2f} M")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# cosine 学习率：T_max 设成 总步数（epoch 数 × 每个 epoch 的 batch 数）
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * 5)


device: cuda
Total parameters: 6.84 M


In [8]:
# 冒烟测试：训练前先验证模型本身没问题（不用真数据，只跑 1 个前向+反向）
_cfg = GPTConfig()
_m = GPT(_cfg)
_fake_x = torch.randint(0, _cfg.vocab_size, (2, 16))
_fake_y = torch.randint(0, _cfg.vocab_size, (2, 16))
_logits, _loss = _m(_fake_x, targets=_fake_y)
print("logits shape:", _logits.shape, "  loss:", _loss.item())
_loss.backward()
print("反向传播 OK，模型结构没问题")
del _cfg, _m, _fake_x, _fake_y, _logits, _loss


logits shape: torch.Size([32, 50257])   loss: 10.918784141540527
反向传播 OK，模型结构没问题


In [9]:
# 训练循环
def train(model, optimizer, scheduler, train_loader, device, epoch):
    model.train()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据移到设备上
        x, y = x.to(device), y.to(device)
        
        # 前向传播
        logits, loss = model(x, targets=y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 调整学习率
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
    return total_loss

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            val_loss += loss.item()
    return val_loss


import os
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(2):
    train_loss = train(model, optimizer, scheduler, train_loader, device, epoch)
    val_loss = eval(model, val_loader, device)
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch}.pt')
    

Epoch: 0, Batch: 0, Loss: 10.8614
Epoch: 0, Batch: 100, Loss: 6.5899
Epoch: 0, Batch: 200, Loss: 6.2080
Epoch: 0, Train Loss: 6.9329, Val Loss: 5.8390
Epoch: 1, Batch: 0, Loss: 5.8190
Epoch: 1, Batch: 100, Loss: 5.7809
Epoch: 1, Batch: 200, Loss: 5.5759
Epoch: 1, Train Loss: 5.4929, Val Loss: 5.2561


In [10]:
# ============== 用训练好的模型"聊天"（其实是续写）==============
# 重要：这是 *预训练模型*，只会按概率续写文本，不会真正一问一答。
# 给它一个开头，它会顺着语料里学到的模式接下去。
# 想要 ChatGPT 那样的对话能力，需要再做指令微调（SFT/RLHF）——那是下一阶段。

model.eval()

# random_split 会把 dataset 包一层，要 .dataset 才能拿到底层 MyDataset 的方法
underlying = train_dataset.dataset if hasattr(train_dataset, 'dataset') else train_dataset

# 莎士比亚风格的开头（input.txt 里随处可见这种"角色名:\n台词"格式）
prompt = "First Citizen:\nBefore we"
ids = underlying.encode(prompt)
x = torch.tensor([ids], dtype=torch.long, device=device)

with torch.no_grad():
    out = model.generate(x, max_new_token=100)

print("prompt :", prompt)
print("续写后 :", underlying.decode(out[0].tolist()))

prompt : First Citizen:
Before we
续写后 : First Citizen:
Before weradical, and she London dead'sANO:
d d Pompe freely come on:
 Sud'd butUDbrid you do more dese theyLV to the oldY:
Then Europe me;
My.
And and much prophecy
 C Templateiansest it view death.
From moring the not my kill home! marble death there not great mist; nothing of vile lustilst I presc:

Ay!First parting
ROMUS:
F1800 and my mind,
